In [ ]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v1_0 import BioJepa, BioJepaConfig
from training_v1_0 import create_model, maybe_compile
from config_v1_0 import DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [ ]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = False
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/jepa/v1_0').expanduser()
ref_root = Path('~/data/jepa/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir = ref_root,
    eval_results_dir=data_root / 'eval_results'
)

## Stage 0: Initialize Model 

In [ ]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=5.699,
    film_linear_multiple=0.6769,
    sim_coeff=50.18,
    std_coeff=25.44,            # sim_coeff * std_to_sim_ratio (0.5069)
    cov_coeff=0.5158,           # sim_coeff * cov_to_sim_ratio (0.01028)
    pert_latent_dim=128,
    pert_mode_dim=64,
    predictor_embed_dim=128,
    predictor_n_layer=4,
    predictor_heads=4,
)


EVAL_BATCH_SIZE = 64

In [ ]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

In [ ]:
USE_COMPILE

### Load Model 

In [ ]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v1_0_ac_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

## Stage 1: Eval
### Alignment Training Eval

In [ ]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
align_eval_results = run_composer_evals(align_eval_ctx)

In [ ]:
save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

In [ ]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()


### Encoder Training Evals

In [ ]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
encoder_eval_results = run_encoder_evals(eval_ctx)

In [ ]:
save_report(encoder_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
encoder_eval_results

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Full Model Eval

In [ ]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v1_0_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

In [ ]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED,
})
full_eval_results = run_ac_evals(eval_ctx)

In [ ]:
save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()